# Procesamiento de datos sobre canciones

- Inteligencia Artificial - grupo 02
- Tarea 3, parte 2. 
- Anthony Sánchez, Andrés Camacho, Enrique Ramírez
- II-2026



In [25]:
import re
import unicodedata
from collections import Counter
import pandas as pd
import math


In [26]:

def preprocesar(texto):
    # pasar a minuscula
    texto = str(texto).lower()
    # quitar tildes
    texto = unicodedata.normalize('NFKD', texto) # esto y la linea que sigue se lo pregunté a claude
    texto = texto.encode('ascii', 'ignore').decode('ascii')
    # retorna terminos a-z
    return re.findall(r'[a-z]+', texto)

def bolsa_de_palabras(texto):
    return Counter(preprocesar(texto)) #esto es un diccionario que cuenta cuantas veces sale una palabra

In [27]:
# Leer hojas de entrenamiento y prueba
train = pd.read_excel('reggaeton_baladas.xlsx', sheet_name='Entrenamiento')
test  = pd.read_excel('reggaeton_baladas.xlsx', sheet_name='Testing')
# Ponerle nombre a columnas
train.columns = ['categoria', 'letra']
test.columns = ['categoria', 'letra']

#Conseguir tokens
def tokenizar(df):
    resultados = []
    for letra in df['letra']:
        resultados.append(preprocesar(letra))
    df['tokens'] = resultados

tokenizar(train)
tokenizar(test)


#visualizar resultado
print('Tamaño de train:', train.shape)  
print('Tamaño de test:', test.shape)
print(train.head())     




Tamaño de train: (552, 3)
Tamaño de test: (138, 3)
   categoria                                              letra  \
0     BALADA  Vuelve a mí\nQue solo y triste me quedé sin ti...   
1     BALADA  Si aún no me lo crees, amor Y quieres, tú, cor...   
2     BALADA  Por cada beso tuyo\nUn sufrimiento recibí sin ...   
3  REGAETTON  Mi bicho anda fugao' y yo quiero que tú me lo ...   
4  REGAETTON  Aquí pasamos moñas por el TSA\nLas putas se mo...   

                                              tokens  
0  [vuelve, a, mi, que, solo, y, triste, me, qued...  
1  [si, aun, no, me, lo, crees, amor, y, quieres,...  
2  [por, cada, beso, tuyo, un, sufrimiento, recib...  
3  [mi, bicho, anda, fugao, y, yo, quiero, que, t...  
4  [aqui, pasamos, monas, por, el, tsa, las, puta...  


In [28]:
# Vocabulario con datos de entrenamiento
# uso set porque no guarda repetidos
def crear_vocabulario(lista_tokens):
    vocabulario = set()
    for tokens in lista_tokens:
        vocabulario.update(tokens)
    return sorted(vocabulario)


vocabulario = crear_vocabulario(train['tokens'])

V = len(vocabulario)

print('Tamaño del vocabulario:', V)
print(vocabulario[:20])


Tamaño del vocabulario: 4158
['a', 'aa', 'ababache', 'abajo', 'abandone', 'abatir', 'abayarde', 'abecedario', 'abierta', 'abiertas', 'abismo', 'abordo', 'abrace', 'abraza', 'abrazado', 'abrazame', 'abrazan', 'abrazando', 'abrazandote', 'abrazare']


In [29]:
# entrenar modelo
def entrenar(df):
    vocabulario = set(crear_vocabulario(df['tokens']))
    V = len(vocabulario)

    clases = ['BALADA', 'REGAETTON']
    priors = {}
    conteos = {}
    total_palabras = {}

    # se filtran las canciones para las dos clases
    for c in clases:
        canciones_c = df[df['categoria'] == c]

        # P(c) = canciones en c / total canciones
        priors[c] = len(canciones_c) / len(df)

        # cuántas veces aparece una palabra en c
        conteos[c] = Counter()
        for tokens in canciones_c['tokens']:
            conteos[c].update(tokens)

        # total de palabras en una clase c
        total_palabras[c] = sum(conteos[c].values())

    # Este return lo hice con claude
    return {'clases': clases, 'vocabulario': vocabulario, 'V': V,
            'priors': priors, 'conteos': conteos, 'total_palabras': total_palabras}


In [30]:
# P(w|c), se usa Laplace
def prob_palabra(modelo, palabra, c):
    conteo = modelo['conteos'][c][palabra]   # por ejemplo, modelo['conteos']['BALADA']['amor']
                                              #  es la cantidad de veces que aparece "amor" en baladas
    return (conteo + 1) / (modelo['total_palabras'][c] + modelo['V'])
# por ejemplo modelo['total_palabras']['BALADA'] es la cantidad de palabras en todas las baladas, incluidas repeticiones


In [31]:
def clasificar(modelo, tokens):
    puntajes = {}
    for c in modelo['clases']:
        # log del prior
        puntaje = math.log(modelo['priors'][c])

        # sumar log de P(w|c) de cada palabra de una cancion
        for w in tokens:
            if w in modelo['vocabulario']: #ingorar palabras que no se conocen
                puntaje += math.log(prob_palabra(modelo, w, c))

        puntajes[c] = puntaje

    # la clase con el puntaje más alto
    prediccion = max(puntajes, key=puntajes.get)
    return prediccion, puntajes


In [ ]:
modelo = entrenar(train)

# probar con tres primeras canciones de conjunto de entrenamiento
for i in range(3):
    pred, punt = clasificar(modelo, train['tokens'][i])
    print('Real:', train['categoria'][i], '| Predicción:', pred, punt)

# frases de inventatas
print(clasificar(modelo, preprocesar('te amo mi amor, mi corazón es tuyo')))
print(clasificar(modelo, preprocesar('dale mami perreo hasta abajo')))


Real: BALADA | Predicción: BALADA {'BALADA': -239.98368629708105, 'REGAETTON': -272.0533218817886}
Real: BALADA | Predicción: BALADA {'BALADA': -233.51553648631537, 'REGAETTON': -246.5542862043641}
Real: BALADA | Predicción: BALADA {'BALADA': -234.43493882230555, 'REGAETTON': -255.42957380203765}
('BALADA', {'BALADA': -43.599100657595564, 'REGAETTON': -50.62792887056353})
('REGAETTON', {'BALADA': -45.90416189311212, 'REGAETTON': -34.350585614685144})
